# ACI-DQN 数据中心调度 —— 云端训练脚本

本 Notebook 用于在云端（Colab / Kaggle / AutoDL 等）训练三种 RL 调度策略：
- **DQN**（标准深度 Q 网络）
- **ACI-DQN**（自适应共形推理增强 DQN）
- **DtACI-DQN**（动态调谐 ACI + 安全动作盾）

训练完成后，模型保存到 `outputs/models/`，可直接下载用于本地评估。

## 1. 环境配置

In [ ]:
# 安装依赖（按需取消注释）
# !pip install numpy pandas matplotlib torch pyyaml scikit-learn

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import torch

# 确保项目根目录在 sys.path
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

from src.utils import load_config, ensure_dir, get_logger, set_global_seed
from src.data_preprocess import run as preprocess_run
from src.datacenter_env import DataCenterEnv, action_to_n, n_to_action
from src.rl.dqn_agent import DQNAgent
from src.rl.train_dqn import (
    EpisodeStats,
    IdentityAugmenter,
    evaluate,
    train_agent,
    rollout_episode,
)
from src.rl.augmenters import ConformalAugmenter
from _common import build_env_and_splits, calibration_arrival_data

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. 加载配置与数据预处理

In [ ]:
# ============================================================
# 配置区域 —— 按需修改
# ============================================================

CONFIG_PATH = "config.yaml"
SEED = 2024

# 训练参数（可覆盖 config.yaml 中的默认值）
TRAIN_EPISODES = 500       # 训练 episode 数
METHODS_TO_TRAIN = ["dqn", "aci_dqn", "dtaci_dqn"]  # 选择要训练的方法
# METHODS_TO_TRAIN = ["aci_dqn"]  # 只训练 ACI-DQN

# ============================================================
cfg = load_config(CONFIG_PATH)
set_global_seed(SEED)

print(f"Config loaded from: {CONFIG_PATH}")
print(f"Train episodes: {TRAIN_EPISODES}")
print(f"Methods: {METHODS_TO_TRAIN}")

In [ ]:
# 数据预处理（生成 processed_load.csv 等文件）
print("=" * 60)
print("Stage 1: Data Preprocessing")
print("=" * 60)

preprocess_run(cfg)
print("Preprocessing done.")

## 3. 构建环境与数据划分

In [ ]:
print("=" * 60)
print("Stage 2: Building Environment and Data Splits")
print("=" * 60)

env, splits, norm_matrix = build_env_and_splits(cfg)

print(f"  Observation dim: {env.observation_dim}")
print(f"  Action dim (discrete bins): {env.action_dim}")
print(f"  Server range: [{env.Nmin}, {env.Nmax}]")
print(f"  Train days: {len(splits['train'])}")
print(f"  Calibration days: {len(splits['calibration'])}")
print(f"  Test days: {len(splits['test'])}")

## 4. 训练三种 RL 方法

训练循环每次都：
1. 随机抽样一个训练日
2. 在该日上执行 roll-out（ε-greedy 探索）
3. 将经验存入 replay buffer
4. 从 buffer 采样 mini-batch 做梯度更新
5. 衰减 epsilon

共训练 `TRAIN_EPISODES` 个 episode。

In [ ]:
def build_rl_agent(method: str, cfg: Dict, env: DataCenterEnv):
    """构建 DQNAgent 及其 StateAugmenter。"""
    rl_cfg = cfg["rl"]

    if method == "dqn":
        aug = IdentityAugmenter()
        state_dim = env.observation_dim  # 17
    elif method in ("aci_dqn", "dtaci_dqn"):
        learner = "aci" if method == "aci_dqn" else "dtaci"
        use_shield = (method == "dtaci_dqn")
        aug = ConformalAugmenter(cfg, learner=learner, use_shield=use_shield)
        state_dim = env.observation_dim + 2 * cfg["qos"]["K"]  # 17 + 2*3 = 23
    else:
        raise ValueError(f"Unknown method: {method}")

    agent = DQNAgent(
        state_dim=state_dim,
        action_dim=env.action_dim,
        hidden=rl_cfg["hidden_sizes"],
        lr=rl_cfg["lr"],
        gamma=rl_cfg["gamma"],
        batch_size=rl_cfg["batch_size"],
        replay_size=rl_cfg["replay_size"],
        epsilon_start=rl_cfg["epsilon_start"],
        epsilon_end=rl_cfg["epsilon_end"],
        epsilon_decay=rl_cfg["epsilon_decay"],
        target_update_interval=rl_cfg["target_update_interval"],
        learning_starts=rl_cfg["learning_starts"],
        reward_scale=rl_cfg["reward_scale"],
        max_grad_norm=rl_cfg["max_grad_norm"],
    )
    return agent, aug


# 训练结果存储
all_histories: Dict[str, List] = {}
all_extra: Dict[str, Dict] = {}

models_dir = ensure_dir(cfg["paths"]["models_dir"])
outputs_dir = ensure_dir(cfg["paths"]["outputs_dir"])
logs_dir = ensure_dir(cfg["paths"]["logs_dir"])

t_total_start = time.time()

for method in METHODS_TO_TRAIN:
    print("\n" + "=" * 60)
    print(f"Training: {method.upper()}")
    print("=" * 60)

    agent, aug = build_rl_agent(method, cfg, env)
    print(f"  State dim: {agent.state_dim}")
    print(f"  Augmenter: {aug.name}")
    print(f"  Device: {agent.device}")

    # ---- Warm up conformal learners on calibration data ----
    if method in ("aci_dqn", "dtaci_dqn"):
        y_hat_cal, y_cal = calibration_arrival_data(
            cfg, norm_matrix, splits["calibration"]
        )
        aug.cp.warm_up_from_calibration(y_hat_cal, y_cal)
        print(f"  Warmed up conformal learners on {len(splits['calibration'])} cal days")

    rng = np.random.default_rng(SEED + {"dqn": 3, "aci_dqn": 4, "dtaci_dqn": 5}[method])

    # ---- Train ----
    t_start = time.time()
    log_path = str(logs_dir / f"{method}_train.log")
    history = train_agent(
        env, agent, aug, splits["train"],
        episodes=TRAIN_EPISODES,
        rng=rng,
        log_path=log_path,
    )
    elapsed = time.time() - t_start
    print(f"  Training completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")

    # ---- Save model ----
    model_path = models_dir / f"{method}.pt"
    agent.save(str(model_path))
    print(f"  Model saved to {model_path}")

    # ---- Save training history ----
    history_df = pd.DataFrame(history)
    history_path = outputs_dir / f"{method}_training_history.csv"
    history_df.to_csv(history_path, index=False)
    print(f"  Training history -> {history_path}")

    all_histories[method] = history
    if hasattr(aug, "metrics"):
        all_extra[method] = aug.metrics()

total_elapsed = time.time() - t_total_start
print(f"\nAll training done in {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")

## 5. 快速评估（可选）

在少量测试日上评估每个模型，输出成本分解和 SLA 违约统计。

In [ ]:
from src.evaluation.metrics import episode_stats_to_row

QUICK_EVAL_DAYS = splits["test"][:10]  # 只用前 10 个测试日快速评估

print("=" * 60)
print("Quick Evaluation (first 10 test days)")
print("=" * 60)

eval_results = []
for method in METHODS_TO_TRAIN:
    agent, aug = build_rl_agent(method, cfg, env)

    # 加载刚训练的模型
    model_path = models_dir / f"{method}.pt"
    agent.load(str(model_path))
    agent.epsilon = 0.0  # greedy evaluation

    # 对 conformal 方法做校准 warm-up
    if method in ("aci_dqn", "dtaci_dqn"):
        y_hat_cal, y_cal = calibration_arrival_data(
            cfg, norm_matrix, splits["calibration"]
        )
        aug.cp.warm_up_from_calibration(y_hat_cal, y_cal)

    eval_rng = np.random.default_rng(SEED + 30)
    stats_list = evaluate(env, agent, aug, QUICK_EVAL_DAYS,
                           rng=eval_rng, record_actions=False)

    for s in stats_list:
        eval_results.append(episode_stats_to_row(method, s))

# 汇总打印
eval_df = pd.DataFrame(eval_results)
summary = eval_df.groupby("method")[[
    "total_objective_cost", "electricity_cost", "qos_cost",
    "switching_cost", "average_active_servers", "average_utilization",
    "P1_sla_violation_rate", "P2_sla_violation_rate", "P3_sla_violation_rate"
]].mean()

print("\n" + summary.to_string())
print("\n(完整评估请运行 python main.py --all)")

## 6. 下载模型文件

训练好的模型在 `outputs/models/` 目录下，压缩后下载。

In [ ]:
# 将模型和训练历史打包
import shutil

archive_name = "aci_dqn_models"
shutil.make_archive(archive_name, "zip", "outputs")
print(f"Archive created: {archive_name}.zip")

# Colab 用户可取消注释：
# from google.colab import files
# files.download(f"{archive_name}.zip")

## 重要说明

### 切换为真实 Trace 数据

当需要接入真实集群 trace 数据时，修改 `config.yaml` 中的：
```yaml
workload:
  source: trace          # 改为 "trace"
  trace_dir: "trace_dataset/"  # trace 数据目录
```
并将 Alibaba PAI trace 的 CSV 文件放入对应目录，程序将自动加载真实任务。

### 压力测试

训练完成后，可运行压力测试评估分布偏移下的鲁棒性：
```bash
python stress_test_scenarios.py
```

### 全量实验
```bash
python main.py --all           # 训练 + 评估全部 6 个 baseline
python generate_figures.py     # 生成论文图表
```